In [0]:
CREATE TABLE IF NOT EXISTS sandbox.z_jungryo_lee.voc_wls_risk_alert_summary (
    group_dim STRING,
    group_key STRING,
    y_feature STRING,
    alert_type STRING,
    alert_priority INT,
    alert_text STRING,
    alert_value DOUBLE,
    created_at TIMESTAMP
) USING DELTA;

INSERT OVERWRITE sandbox.z_jungryo_lee.voc_wls_risk_alert_summary
WITH filtered_coef AS (
    SELECT
        group_dim,
        group_key,
        y_feature,
        x_feature,
        coef,
        p_value
    FROM sandbox.z_jungryo_lee.voc_wls_coef_summary    WHERE p_value < 0.1
      AND ABS(coef) >= 0.1
),
negative_driver_alert AS (
    SELECT
        group_dim,
        group_key,
        y_feature,
        'NEGATIVE_DRIVER' AS alert_type,         1 AS alert_priority,
        CONCAT(
            '⚠ ',
            group_key,
            ' → ',
            x_feature,
            ' (',
            CAST(ROUND(coef, 2) AS STRING),
            ')'
        ) AS alert_text,
        coef AS alert_value,
        CURRENT_TIMESTAMP() AS created_at
    FROM filtered_coef
    WHERE coef < 0
),

low_r2_alert AS (
    SELECT
        group_dim,
        group_key,
        y_feature,
        'LOW_R2' AS alert_type,
        2 AS alert_priority,
        CONCAT(
            '⚠ ',
            group_key,
            ' → Low R² (',
            CAST(ROUND(r_squared, 2) AS STRING),
            ')'
        ) AS alert_text,

        r_squared AS alert_value,
        CURRENT_TIMESTAMP() AS created_at
    FROM sandbox.z_jungryo_lee.voc_wls_model_summary
    WHERE r_squared < 0.40
),
driver_count_summary AS (
    SELECT
        group_dim,
        group_key,
        y_feature,
        COUNT(*) AS valid_driver_cnt
    FROM filtered_coef     
    GROUP BY group_dim, group_key, y_feature
),

low_driver_count_alert AS (
    SELECT
        m.group_dim,
        m.group_key,
        m.y_feature,
        'LOW_DRIVER_COUNT' AS alert_type,
        3 AS alert_priority,
        CONCAT(
            '⚠ ',
            m.group_key,
            ' → Significant drivers are limited (',
            CAST(COALESCE(d.valid_driver_cnt, 0) AS STRING),
            ')'
        ) AS alert_text,
        CAST(COALESCE(d.valid_driver_cnt, 0) AS DOUBLE) AS alert_value,
        CURRENT_TIMESTAMP() AS created_at
    FROM sandbox.z_jungryo_lee.voc_wls_model_summary m
    LEFT JOIN driver_count_summary d
        ON m.group_dim = d.group_dim
       AND m.group_key = d.group_key
       AND m.y_feature = d.y_feature
    WHERE COALESCE(d.valid_driver_cnt, 0) <= 2
),

high_cond_no_alert AS (
    SELECT
        group_dim,         
        group_key,
        y_feature,
        'HIGH_COND_NO' AS alert_type,
        4 AS alert_priority,
        CONCAT(
            '⚠ ',
            group_key,
            ' → High multicollinearity risk (Cond No = ',
            CAST(ROUND(cond_no, 0) AS STRING),
            ')'
        ) AS alert_text,
        cond_no AS alert_value,
        CURRENT_TIMESTAMP() AS created_at
    FROM sandbox.z_jungryo_lee.voc_wls_model_summary
    WHERE cond_no > 300
),
low_review_count_alert AS (
    SELECT
        group_dim,
        group_key,
        y_feature,
        'LOW_REVIEW_COUNT' AS alert_type,
        5 AS alert_priority,         
        CONCAT(
            '⚠ ',
            group_key,
            ' → Low review count (',
            CAST(y_obs AS STRING),
            ')'
        ) AS alert_text,
        CAST(y_obs AS DOUBLE) AS alert_value,
        CURRENT_TIMESTAMP() AS created_at
    FROM sandbox.z_jungryo_lee.voc_wls_model_summary
    WHERE y_obs < 100
)
SELECT * FROM negative_driver_alert
UNION ALL
SELECT * FROM low_r2_alert
UNION ALL
SELECT * FROM low_driver_count_alert
UNION ALL
SELECT * FROM high_cond_no_alert 
UNION ALL
SELECT * FROM low_review_count_alert;

 select *
 from sandbox.z_jungryo_lee.voc_wls_risk_alert_summary